# Fashion-MNIST: Shallow vs Deep Networks
## Depth Study — 2-Layer vs 4-Layer vs 8-Layer MLPs

**Course:** Deep Learning I &nbsp;|&nbsp; **Assignment:** Section 2 — Network Depth Experiments  
**Dataset:** Fashion-MNIST &nbsp;|&nbsp; **Framework:** PyTorch

---

### Research Questions

| ID | Question |
|----|----------|
| **RQ1** | Does increasing depth improve performance under a fixed parameter budget? |
| **RQ2** | How does depth affect gradient flow? |
| **RQ3** | What role does activation choice play? |
| **RQ4** | Can BatchNorm restore trainability? |

---

> **Notebook usage:** Run all cells top-to-bottom. Set `FAST_MODE = True` for a quick
> test run (~2 min on CPU) or `FAST_MODE = False` for the full experiment (~8–15 min on GPU).
> `AUTO_RESUME = True` (default) skips any run whose results already exist in the CSV.

In [ ]:
# ── Cell 1: Dependency Installation ──────────────────────────────────────────
# Run this cell first. On Colab all packages are pre-installed;
# on a fresh local env this installs any missing packages.

import importlib, subprocess, sys

_required = [
    ("torch",       "torch"),
    ("torchvision", "torchvision"),
    ("numpy",       "numpy"),
    ("pandas",      "pandas"),
    ("matplotlib",  "matplotlib"),
    ("seaborn",     "seaborn"),
]

for _import_name, _pkg_name in _required:
    if importlib.util.find_spec(_import_name) is None:
        print(f"Installing {_pkg_name}...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", _pkg_name],
            check=True
        )

print("All packages ready.")

In [ ]:
# ── Cell 2: Project Root Discovery + Core Imports ────────────────────────────
# Locates the project root directory (the one containing src/) and adds it
# to sys.path. Works whether the notebook is run from notebooks/ or the root.

import os, sys

_candidates = [
    os.getcwd(),
    os.path.join(os.getcwd(), ".."),
    os.path.dirname(os.getcwd()),
]
PROJECT_ROOT = None
for _p in _candidates:
    if os.path.isdir(os.path.join(_p, "src")):
        PROJECT_ROOT = os.path.abspath(_p)
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot find project root (directory containing src/).\n"
        "Ensure the notebook lives inside fashion-mnist-depth-study/ and "
        "that src/ exists at the project root."
    )

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)   # Relative paths in CONFIG resolve from here
print(f"Project root : {PROJECT_ROOT}")

import platform
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# src/ imports
from src.utils  import (
    CONFIG, CLASS_NAMES, set_seed,
    get_dataloaders, load_results, load_summary,
    compute_aggregate_summary,
)
from src.models import ConfigurableMLP
from src.train  import ExperimentRunner
from src.plots  import (
    plot_val_accuracy_curves,
    plot_val_loss_curves,
    plot_test_accuracy_bar,
    plot_gradient_norm_by_layer,
    plot_gradient_norm_over_epochs,
    plot_activation_heatmap,
    plot_batchnorm_recovery,
    generate_all_figures,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device         : {DEVICE}")

# DataLoader workers: Windows Jupyter requires num_workers=0
NUM_WORKERS = 0 if platform.system() == "Windows" else 2
print(f"num_workers    : {NUM_WORKERS}")

In [ ]:
# ── Cell 3: Experiment Configuration ─────────────────────────────────────────
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ONLY EDIT THIS CELL to change experiment settings              ║
# ╚══════════════════════════════════════════════════════════════════╝

FAST_MODE   = False   # True → 1 seed, 10 epochs (~2 min CPU)  |  False → 3 seeds, 50 epochs
AUTO_RESUME = True    # True → skip runs already in results.csv (safe for Colab disconnects)

# Apply FAST_MODE overrides to CONFIG
if FAST_MODE:
    CONFIG["epochs"] = CONFIG["fast_epochs"]   # 10 epochs
    CONFIG["seeds"]  = CONFIG["fast_seeds"]    # [42]
    print(f"FAST_MODE ON  → epochs={CONFIG['epochs']}, seeds={CONFIG['seeds']}")
else:
    print(f"Full mode     → epochs={CONFIG['epochs']}, seeds={CONFIG['seeds']}")

# Create output directories
for _dir in [CONFIG["data_dir"], Path(CONFIG["results_path"]).parent,
             CONFIG["figures_dir"]]:
    Path(_dir).mkdir(parents=True, exist_ok=True)

print(f"\nCONFIG summary:")
print(f"  batch_size  : {CONFIG['batch_size']}")
print(f"  lr          : {CONFIG['lr']}")
print(f"  depths      : {CONFIG['depths']}")
print(f"  widths      : {CONFIG['widths']}")
print(f"  activations : {CONFIG['activations']}")
print(f"  results  →  {CONFIG['results_path']}")
print(f"  figures  →  {CONFIG['figures_dir']}")

---
## Section 1: Introduction

### Problem Statement

Depth is a central hyperparameter in neural network design. Deep learning theory suggests that deeper networks can represent more complex functions using fewer parameters than shallow networks (Telgarsky, 2016). However, increasing depth introduces two practical challenges:

1. **Vanishing Gradients**: Gradients shrink exponentially as they propagate backward through many layers, making early layers unable to learn. This is especially severe with sigmoid activations, where σ'(x) ≤ 0.25 per layer → cumulative attenuation of 0.25^8 ≈ 10⁻⁵ after 8 layers.

2. **Width-Depth Trade-off**: Under a fixed parameter budget, deeper networks require narrower layers. Narrower layers may restrict representational capacity at each level.

### This Study

We compare MLPs of depth 2, 4, and 8 on Fashion-MNIST under a fixed parameter budget of ~500K, systematically investigating:

| RQ | Question | Experiment |
|----|----------|------------|
| RQ1 | Does depth improve performance? | Exp 1 (ReLU, all depths) |
| RQ2 | How does depth affect gradient flow? | Exp 2 (gradient norms) |
| RQ3 | What role does activation play? | Exp 2 (ReLU vs Sigmoid) |
| RQ4 | Can BatchNorm restore trainability? | Exp 3 (8L Sigmoid ±BN) |

---
## Section 2: Dataset — Fashion-MNIST

In [ ]:
# ── Cell 6: Load Dataset ──────────────────────────────────────────────────────
set_seed(42)
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir    = CONFIG["data_dir"],
    batch_size  = CONFIG["batch_size"],
    val_size    = CONFIG["val_size"],
    train_size  = CONFIG["train_size"],
    norm_mean   = CONFIG["norm_mean"],
    norm_std    = CONFIG["norm_std"],
    seed        = 42,
    num_workers = NUM_WORKERS,
)

print("Fashion-MNIST dataset loaded.")
print(f"  Train      : {len(train_loader.dataset):,} samples")
print(f"  Validation : {len(val_loader.dataset):,} samples")
print(f"  Test       : {len(test_loader.dataset):,} samples")
print(f"  Classes    : {len(CLASS_NAMES)} → {CLASS_NAMES}")
print(f"  Input dim  : 28 × 28 = 784 features (after flatten)")
print(f"  Normalisation: mean={CONFIG['norm_mean']}, std={CONFIG['norm_std']}")
print(f"    Pixel range: [0,1] → [{-CONFIG['norm_mean']/CONFIG['norm_std']:.1f}, "
      f"{(1-CONFIG['norm_mean'])/CONFIG['norm_std']:.1f}]")

In [ ]:
# ── Cell 7: Visualise Dataset Samples ────────────────────────────────────────
from torchvision import datasets, transforms

raw_ds = datasets.FashionMNIST(
    CONFIG["data_dir"], train=True, download=False,
    transform=transforms.ToTensor()
)

fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
shown = {i: 0 for i in range(10)}
count = 0

for img, label in raw_ds:
    r = shown[label]
    if r < 2:
        axes[r, label].imshow(img.squeeze(), cmap="gray", vmin=0, vmax=1)
        axes[r, label].axis("off")
        if r == 0:
            axes[r, label].set_title(CLASS_NAMES[label], fontsize=7.5)
        shown[label] += 1
        count += 1
    if count == 20:
        break

fig.suptitle("Fashion-MNIST: 2 Raw Samples per Class (Before Normalisation)",
             fontsize=12, y=1.03)
plt.tight_layout()
Path(CONFIG["figures_dir"]).mkdir(parents=True, exist_ok=True)
fig.savefig(f"{CONFIG['figures_dir']}/dataset_samples.png", dpi=150, bbox_inches="tight")
plt.show()

print("Note: During training, images are normalised to [-1, 1] (mean=0.5, std=0.5).")

---
## Section 3: Architecture Design

### Equal Parameter Budget (~500K)

To ensure a **fair comparison**, all models are constrained to approximately the same number of trainable parameters (~500K). This is achieved by reducing layer width as depth increases:

| Depth | Width | Parameter Formula |
|-------|-------|------------------|
| 2L    | 413   | 784×413 + 1×413² + 413×10 |
| 4L    | 296   | 784×296 + 3×296² + 296×10 |
| 8L    | 215   | 784×215 + 7×215² + 215×10 |

> **Why this matters:** Without controlling parameters, deeper networks would have more total capacity. Any accuracy difference would be attributable to *size* rather than *depth*.

### Layer Template

```
Input (784)
  → Flatten
  → [ Linear(in, W) → [BatchNorm1d(W)] → Activation ] × depth
  → Linear(W, 10)           ← output logits (no activation)
```

BatchNorm is **off** for Experiments 1 & 2 (to isolate depth/activation effects)  
BatchNorm is **on** only for the 8L Sigmoid model in Experiment 3.

In [ ]:
# ── Cell 9: Instantiate Models and Display Architectures ─────────────────────
print("Model architectures (ReLU, no BatchNorm):")

for depth in CONFIG["depths"]:
    width = CONFIG["widths"][depth]
    m = ConfigurableMLP(
        input_dim=CONFIG["input_dim"],
        num_classes=CONFIG["num_classes"],
        depth=depth,
        width=width,
        activation="ReLU",
        use_batchnorm=False,
    )
    print(f"\n{'='*55}")
    print(f"  {m}")
    print(f"  Network:")
    for i, layer in enumerate(m.network):
        print(f"    [{i}] {layer}")

In [ ]:
# ── Cell 10: Parameter Count Table ───────────────────────────────────────────
rows = []
for depth in CONFIG["depths"]:
    width = CONFIG["widths"][depth]
    m_no_bn = ConfigurableMLP(
        depth=depth, width=width, activation="ReLU", use_batchnorm=False
    )
    rows.append({
        "Model"     : f"{depth}L MLP (ReLU)",
        "Depth"     : depth,
        "Width"     : width,
        "Parameters": m_no_bn.count_parameters(),
        "Diff from 500K": m_no_bn.count_parameters() - 500_000,
    })

# BatchNorm overhead
m_bn  = ConfigurableMLP(depth=8, width=215, activation="Sigmoid", use_batchnorm=True)
m_nbn = ConfigurableMLP(depth=8, width=215, activation="Sigmoid", use_batchnorm=False)
bn_overhead = m_bn.count_parameters() - m_nbn.count_parameters()

param_df = pd.DataFrame(rows)
print("Parameter Count Table:")
print(param_df.to_string(index=False))
print(f"\nBatchNorm overhead (8L Sigmoid, width=215):")
print(f"  +{bn_overhead:,} params  (= 2 × width × depth = 2 × 215 × 8)")
print(f"  Fraction of total: {bn_overhead/m_bn.count_parameters()*100:.2f}%  (negligible)")
print("\n✓ Widths adjusted so parameter count is approximately constant across depths.")

---
## Section 4: Experiment 1 — Depth Comparison (ReLU)

**Research Question (RQ1):** Does increasing depth improve performance under a fixed parameter budget?

**Setup:**
- Models: 2L, 4L, 8L MLP — all ReLU, no BatchNorm  
- Seeds: 42, 123, 7 (or [42] in FAST_MODE)  
- Epochs: 50 (or 10 in FAST_MODE)  
- All models use ~500K parameters  

**Hypothesis:** Deeper models may learn more hierarchical features, but gains may be marginal on Fashion-MNIST — a relatively simple dataset whose discriminative features (texture, silhouette) may be fully captured by a 2-layer representation.

**What to look for in the results:**
- `Fig 1`: Does a deeper model's accuracy curve converge higher or faster?
- `Fig 2`: Does a deeper model's loss converge lower?
- `Fig 3`: Are differences statistically meaningful given ± std error bars?

In [ ]:
# ── Cell 12: Run Experiment 1 ─────────────────────────────────────────────────
# AUTO_RESUME=True: skips any run_id already in results.csv
# Safe to re-run; will not duplicate results.

runner = ExperimentRunner(
    config      = CONFIG,
    fast_mode   = FAST_MODE,
    auto_resume = AUTO_RESUME,
    device      = DEVICE,
    num_workers = NUM_WORKERS,
    verbose     = True,
)

runner.run_experiment_1()
print(f"\nResults saved → {CONFIG['results_path']}")
print(f"Summary saved → {CONFIG['summary_path']}")

In [ ]:
# ── Cell 13: Figure 1 — Validation Accuracy vs Epoch ─────────────────────────
plot_val_accuracy_curves(
    results_path = CONFIG["results_path"],
    figures_dir  = CONFIG["figures_dir"],
    activation   = "ReLU",
    batchnorm    = 0,
)
# Expected output:
# - Three smooth curves (2L, 4L, 8L) with ±1 std shading
# - Curves should reach 80-90% validation accuracy
# - A curve starting higher or converging faster indicates a depth advantage

In [ ]:
# ── Cell 14: Figure 2 — Validation Loss vs Epoch ─────────────────────────────
plot_val_loss_curves(
    results_path = CONFIG["results_path"],
    figures_dir  = CONFIG["figures_dir"],
    activation   = "ReLU",
    batchnorm    = 0,
)
# Expected output:
# - Three decreasing loss curves
# - A model reaching lower final loss demonstrates better calibration

In [ ]:
# ── Cell 15: Figure 3 — Test Accuracy Bar Chart ───────────────────────────────
# Figure 3 requires Experiment 2 (Sigmoid) data. It is generated in Cell 20B below after Experiment 2 completes.
# Expected output:
# - Bar chart with 6 bars (3 depths × 2 activations, no BN)
# - Error bars represent std across seeds
# - Sigmoid bars should be notably lower than ReLU (Experiment 2 data needed)

### Experiment 1 Analysis

**Interpreting the depth comparison (Figs 1–3):**

**If deeper models outperform shallower models:**  
The [N]L MLP achieves higher validation and test accuracy, suggesting Fashion-MNIST contains sufficient hierarchical structure to benefit from deeper representations under equal parameter budgets. The convergence epoch comparison indicates [faster/slower] learning.

**If all depths perform similarly:**  
All three models converge to similar accuracy (~[X]%). This indicates that Fashion-MNIST's discriminative features are fully captured by a 2-layer MLP at the 500K parameter scale. Increasing depth provides no meaningful benefit — the dataset is not complex enough to require hierarchical feature composition. The differences, if any, fall within ± std.

**If the 8L model underperforms:**  
The 8L model shows lower accuracy despite equal parameters. Even with ReLU (which avoids the vanishing gradient problem), deeper networks may encounter harder optimization landscapes. This motivates the gradient flow analysis in Experiment 2.

> **Key takeaway:** The gradient attenuation ratio (Section 7 summary table) provides mechanistic insight beyond accuracy — it quantifies how efficiently training signal reaches early layers.

---
## Section 5: Experiment 2 — Activation Study (Vanishing Gradients)

**Research Questions (RQ2 + RQ3):** How does depth affect gradient flow? What role does activation choice play?

**Setup:**
- Models: 2L, 4L, 8L × {ReLU, Sigmoid} = 6 configurations (no BatchNorm)
- ReLU runs reused from Experiment 1 via AUTO_RESUME (no re-training)

**Theoretical Background:**

Backpropagation computes gradients via the chain rule. Each layer contributes a factor of the activation derivative:

| Activation | Derivative | Consequence |
|-----------|-----------|-------------|
| Sigmoid | σ'(x) = σ(x)(1−σ(x)) ≤ **0.25** | 0.25^8 ≈ **1.5×10⁻⁵** after 8 layers |
| ReLU | f'(x) = **1** (x>0), **0** (x<0) | Gradients pass unchanged through active neurons |

**Primary Figure: Fig 4A** — Gradient Norm vs Layer Index at the final training epoch.  
A decaying curve from right (output) to left (input) visually demonstrates vanishing gradients.

In [ ]:
# ── Cell 18: Run Experiment 2 ─────────────────────────────────────────────────
# ReLU runs are automatically reused (AUTO_RESUME skips existing run_ids)
# Only Sigmoid runs are new.

runner.run_experiment_2()
print(f"\nExperiment 2 complete.")

In [ ]:
# ── Cell 19: Figure 4A — Gradient Norm vs Layer Depth (PRIMARY FIGURE) ───────
plot_gradient_norm_by_layer(
    results_path = CONFIG["results_path"],
    figures_dir  = CONFIG["figures_dir"],
)
# Expected output:
# - 8L ReLU: approximately flat/stable gradient norms across layers 1-8
# - 8L Sigmoid: exponentially decaying gradient norms from layer 8 → layer 1
# - Log-scale y-axis reveals orders-of-magnitude differences
# - Gradient attenuation ratio annotated on each curve

In [ ]:
# ── Cell 20: Figure 4B — Gradient Norm vs Epoch (SUPPLEMENTAL) ───────────────
plot_gradient_norm_over_epochs(
    results_path = CONFIG["results_path"],
    figures_dir  = CONFIG["figures_dir"],
)
# Expected output:
# - Left panel (Layer 1 / input side): Sigmoid grads near zero throughout;
#   ReLU grads significantly larger
# - Right panel (Layer 8 / output side): Both activations show non-trivial
#   gradient — confirms the problem is gradient propagation, not output gradients

In [ ]:
# ── Cell 20B: Figure 3 — Test Accuracy Bar Chart (ALL 6 configs) ──────────────
# Placed here because both ReLU (Exp 1) AND Sigmoid (Exp 2) data are now available.
plot_test_accuracy_bar(
    summary_path = CONFIG["summary_path"],
    figures_dir  = CONFIG["figures_dir"],
)


### Experiment 2 Analysis

**Gradient Norm vs Layer Depth (Fig 4A — Primary Figure):**

This figure is the mechanistic core of the study. Reading from right (output) to left (input):

- **8L ReLU:** Gradient norms remain approximately stable across layers (log scale shows near-horizontal trend). This demonstrates *healthy gradient flow* — every layer receives comparable training signal.

- **8L Sigmoid:** If gradient norms decay from right to left (downward-sloping on log scale), this *directly visualises vanishing gradients*. The slope of the line corresponds to the per-layer attenuation factor.

**Gradient Attenuation Ratio** (first_hidden_layer_grad / last_hidden_layer_grad):
- `8L ReLU` → ratio ≈ [fill from summary table] (ideally close to 1.0)
- `8L Sigmoid` → ratio ≈ [fill from summary table] (expected >> 1)

**Why Adam doesn't fully fix it:**  
Adam's adaptive learning rate scales small gradients up per-parameter, partially compensating. However: (1) direction of updates still depends on gradient sign which becomes unreliable near zero; (2) Adam does not change the *relative* signal strength across layers; (3) the empirical gradient ratio still reveals the disparity.

**Connection to accuracy:**  
If 8L Sigmoid shows dramatically weaker early-layer gradients, early layers fail to specialise. The network essentially behaves like a shallower model with random early-layer features — consistent with any observed accuracy drop.

---
## Section 6: Experiment 3 — BatchNorm Recovery

**Research Question (RQ4):** Can BatchNorm restore trainability to a deep Sigmoid network?

**Setup:**
- Models: 8L Sigmoid (no BN) vs 8L Sigmoid + BatchNorm
- 8L Sigmoid (no BN) runs are reused from Experiment 2 (AUTO_RESUME)

**BatchNorm Mechanism:**

BatchNorm normalises each layer's pre-activations:
```
ẑ = (z − μ_B) / √(σ²_B + ε)     [normalise to zero mean, unit variance]
y = γẑ + β                          [learnable scale and shift]
```

For Sigmoid networks specifically:
- By keeping pre-activations near zero, BN ensures inputs arrive in Sigmoid's **linear region** (near x=0), where σ'(0) = 0.25 — the **maximum** of the sigmoid derivative
- Without BN, deep Sigmoid networks saturate at ±∞ where σ'(x) → 0
- BN breaks the exponential gradient decay by resetting the pre-activation distribution at each layer

In [ ]:
# ── Cell 23: Run Experiment 3 ─────────────────────────────────────────────────
# 8L Sigmoid (no BN) is reused from Experiment 2.
# Only 8L Sigmoid + BN is new (3 seeds).

runner.run_experiment_3()
print(f"\nExperiment 3 complete.")

In [ ]:
# ── Cell 24: Figure 6 — BatchNorm Recovery ────────────────────────────────────
plot_batchnorm_recovery(
    results_path = CONFIG["results_path"],
    figures_dir  = CONFIG["figures_dir"], 
)
# Expected output:
# Panel A: 8L Sigmoid+BN should show substantially higher val_acc than 8L Sigmoid
# Panel B: Gradient attenuation ratio for BN model should be much closer to 1.0
#          than the no-BN Sigmoid model

### Experiment 3 Analysis

**BatchNorm Recovery (Fig 6):**

**Panel A (Validation Accuracy):**  
- If BatchNorm substantially narrows the accuracy gap, it demonstrates the original failure was primarily a *gradient flow problem*, not a *capacity problem*. The network has the representational power — it just couldn't access it without proper gradient signal.
- If the gap persists partially, BatchNorm helps but Sigmoid's saturating nature still imposes training disadvantages vs ReLU.

**Panel B (Gradient Attenuation Ratio):**  
- A dramatically reduced ratio for the BN model confirms that BN directly addresses gradient flow.
- The ratio should trend toward ~1 for the BN model (uniform gradient flow) vs remaining >> 1 without BN.

**The Critical Distinction:**  
BatchNorm does not add significant capacity (+3,440 params out of ~496K = 0.7%). Its entire benefit is mechanistic: it keeps pre-activations in the high-gradient region of Sigmoid. This is a clean causal demonstration of the gradient flow hypothesis.

**Limitation:**  
Even with BN, 8L Sigmoid may not reach 8L ReLU accuracy. ReLU has additional advantages: its derivative is exactly 1 in the active region (not bounded by 0.25), and it does not require normalisation to maintain gradient flow. BatchNorm is a fix, not an equaliser.

---
## Section 7: Summary Table

All results reported as **mean ± std** across [3 seeds (full mode) / 1 seed (FAST_MODE)].

**Column definitions:**
- **Test Accuracy:** Mean ± Std across seeds — final evaluation on 10,000 test samples
- **Convergence Epoch:** First epoch where val_acc ≥ 0.95 × max(val_acc)
- **Gradient Ratio:** `‖∇W_layer1‖ / ‖∇W_layer8‖` at final epoch — higher = more vanishing

In [ ]:
# ── Cell 27: Aggregate Summary Table ─────────────────────────────────────────
agg = compute_aggregate_summary(CONFIG["summary_path"])

if agg.empty:
    print("No results yet — run experiments first.")
else:
    def fmt_mean_std(m, s):
        if pd.isna(s) or s == 0:
            return f"{m:.4f}"
        return f"{m:.4f} ± {s:.4f}"

    display_rows = []
    for _, row in agg.iterrows():
        display_rows.append({
            "Depth"      : int(row["depth"]),
            "Width"      : int(row["width"]),
            "Activation" : row["activation"],
            "BatchNorm"  : "Yes" if row["batchnorm"] else "No",
            "Parameters" : f"{int(row['parameter_count']):,}",
            "Test Acc"   : fmt_mean_std(row["mean_test_acc"], row["std_test_acc"]),
            "Conv.Epoch" : fmt_mean_std(row["mean_convergence_epoch"], row["std_convergence_epoch"]),
            "Grad Ratio" : fmt_mean_std(row["mean_gradient_ratio"], row["std_gradient_ratio"]),
            "Seeds"      : int(row["n_seeds"]),
        })

    table_df = pd.DataFrame(display_rows)
    print("=" * 100)
    print("SUMMARY TABLE: All Results (mean ± std across seeds)")
    print("=" * 100)
    print(table_df.to_string(index=False))
    print("\nGrad Ratio = ‖∇W_layer1‖ / ‖∇W_lastHidden‖  (higher → more vanishing gradient)")

In [ ]:
# ── Cell 28: Figure 5 — Activation Comparison Heatmap ────────────────────────
plot_activation_heatmap(
    summary_path = CONFIG["summary_path"],
    figures_dir  = CONFIG["figures_dir"],
)
# Expected output:
# - 3×2 heatmap (rows=depth, cols=activation)
# - ReLU column cells should be warmer (higher accuracy) than Sigmoid column
# - Colour gradient reveals cross-configuration performance at a glance

---
## Section 8: Discussion

### On Depth and Performance (RQ1)

The depth comparison (Experiment 1) reveals whether hierarchical feature composition matters for Fashion-MNIST at this parameter scale.

**Theoretical framing:** Deep networks can represent certain function classes exponentially more efficiently than shallow networks (Telgarsky, 2016). However, this theoretical advantage requires:
1. The task to genuinely require hierarchical features
2. Sufficient depth to be beyond the shallow network's representation limit
3. Successful optimisation (gradient flow must reach all layers)

Fashion-MNIST's images are simple (28×28 grayscale, coarse textures and silhouettes). A 2L MLP with width=413 (~500K params) may already capture all necessary discriminative structure.

### On Gradient Flow (RQ2 + RQ3)

The gradient norm analysis (Experiment 2, Fig 4A) provides mechanistic evidence for or against vanishing gradients.

**Why ReLU preserves gradients:**
- In the active region (x > 0): f'(x) = 1 exactly → gradient passes unchanged
- Chain rule product: 1^L = 1 → no multiplicative decay
- Only dead neurons (x < 0 always) block flow, but these don't compound

**Why Sigmoid kills gradients:**
- σ'(x) ≤ 0.25 always → chain rule product ≤ 0.25^8 ≈ 10^{-5} for 8 layers
- Maximum gradient at x=0, but saturated neurons push x away from zero
- Kaiming/Xavier init helps initially but doesn't prevent saturation during training

### On BatchNorm (RQ4)

BatchNorm's mechanism is precise: by keeping pre-activations near zero, it ensures Sigmoid inputs remain in the high-gradient region (σ'(0) = 0.25 = maximum). This prevents the cascade of gradient attenuation that makes 8L Sigmoid networks untrainable without BN.

Importantly, BatchNorm's benefit here is **gradient flow improvement**, not capacity improvement — the BN overhead is ~3,440 params out of ~496K total (0.7%).

### Threats to Validity

1. **Task Simplicity:** Fashion-MNIST achieves 90%+ with shallow MLPs. Results may not generalise to harder datasets requiring genuine hierarchical representations.

2. **Width-Depth Confound:** Equal parameter count requires unequal widths. Performance differences may partially reflect width effects rather than depth effects.

3. **Limited Statistical Power:** Three seeds provide rough confidence intervals but insufficient power for formal significance testing. Differences within ±1 std should be treated cautiously.

4. **Generalisation Scope:** Results apply only to fully-connected MLPs. CNNs have built-in translational invariance and local feature extraction — depth-performance relationships differ fundamentally.

---
## Section 9: Conclusion

### Direct Answers to Research Questions

**RQ1: Does increasing depth improve performance under a fixed parameter budget?**  
> *(Fill in after running experiments with actual numbers from the summary table)*  
> Template: "[Yes/No/Marginally]. The [N]L model achieved [X]% ± [Y]% vs [Z]% ± [W]% for the 2L baseline. [Interpretation based on magnitude of difference relative to std.]"

**RQ2: How does depth affect gradient flow?**  
> Depth amplifies gradient attenuation in both ReLU and Sigmoid networks, but the effect is qualitatively different. For ReLU, the gradient attenuation ratio remains approximately [1.X], indicating mild attenuation. For 8L Sigmoid, the ratio is approximately [X.X], confirming that early layers receive gradient signals orders of magnitude weaker than the output-side layers — the quantitative signature of vanishing gradients.

**RQ3: What role does activation choice play?**  
> Activation choice is the primary determinant of trainability in deep networks. ReLU's gradient-preserving property (f'=1 in active region) enables training at depth 8 with no architectural modification. Sigmoid's bounded derivative (≤0.25) makes 8-layer training effectively non-functional via conventional gradient descent, as demonstrated by the gradient norm analysis.

**RQ4: Can BatchNorm restore trainability?**  
> *(Fill in after experiments)*  
> Template: "Yes/Partially. Adding BatchNorm to the 8L Sigmoid model improved test accuracy from [X]% to [Y]% (mean), reducing the gradient attenuation ratio from [A] to [B]. This confirms that the original failure was primarily a gradient flow problem: BN keeps pre-activations in Sigmoid's linear region, preventing gradient decay."

---

### Final Takeaway

This study demonstrates that **depth alone does not guarantee improved performance**. The interaction between depth, activation function, and normalisation strategy determines whether a deep network is effectively trainable. For Fashion-MNIST:

- Depth provides [marginal / no / clear] accuracy benefit under equal parameter budgets
- Activation function is more critical than depth for training stability
- BatchNorm is an effective architectural remedy for vanishing gradients in Sigmoid networks

The key methodological lesson: **understanding why a model fails — through gradient diagnostics — is more informative than reporting accuracy alone.** A model that trains slowly or stagnates due to vanishing gradients requires architectural intervention (activation choice, normalisation), not more data or epochs.

---
## Section 10: Final Execution Summary

This cell aggregates the final metrics, verifies that all outputs were generated successfully, and provides a quick verification printout of the entire pipeline.

In [ ]:
# ── Cell 31: Final Execution Verification ─────────────────────────────────────
import os
from pathlib import Path
import pandas as pd

print("=" * 80)
print("FINAL PIPELINE VERIFICATION")
print("=" * 80)

# 1. Verify Dataset
print("\n[1] DATASET VERIFICATION")
print(f"Train size: {len(train_loader.dataset):,}")
print(f"Val size  : {len(val_loader.dataset):,}")
print(f"Test size : {len(test_loader.dataset):,}")

# 2. Verify Figures
print("\n[2] FIGURES GENERATED")
fig_dir = Path(CONFIG['figures_dir'])
expected_figs = [
    "dataset_samples.png",
    "fig1_val_accuracy.png",
    "fig2_val_loss.png",
    "fig3_test_accuracy.png",
    "fig4a_gradient_norm_layer.png",
    "fig4b_gradient_norm_epoch.png",
    "fig5_activation_heatmap.png",
    "fig6_batchnorm_recovery.png"
]
for f in expected_figs:
    f_path = fig_dir / f
    status = "✅ Found" if f_path.exists() else "❌ Missing"
    print(f"  {status} : {f}")

# 3. Verify Results CSV
print("\n[3] EXPERIMENT RESULTS")
summary_path = Path(CONFIG['summary_path'])
if summary_path.exists():
    df = pd.read_csv(summary_path)
    print(f"  ✅ Found summary CSV with {len(df)} total runs recorded.")
    print("\n  High-Level Test Accuracies (Averaged across seeds):")
    agg = compute_aggregate_summary(summary_path)
    for _, row in agg.iterrows():
        bn_str = "+ BN" if row['batchnorm'] else ""
        print(f"    - {int(row['depth'])}L {row['activation']} {bn_str:>4} : "
              f"{row['mean_test_acc']:.4f} ± {row['std_test_acc']:.4f}")
else:
    print("  ❌ Missing summary CSV!")

print("\n" + "=" * 80)
print("PIPELINE COMPLETE. Notebook successfully executed top-to-bottom.")
print("=" * 80)
